# Train an RVC voice model

Trains a voice model on Colab's free GPU with the RVC WebUI. It uses the same RVC version as the
GitHub Action in this repo, so the models it produces work there.

**Before you start**
1. In the menu choose **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive, make a folder `RVC/dataset/<your-model-name>/` (for example `RVC/dataset/MyVoice/`)
   and upload the voice recordings there: **10–30 minutes of clean speech or singing**, one voice only,
   no music, no background noise, no reverb. Several files are fine (wav, mp3, flac…).

Then run the cells below in order (click ▶ on each).
Everything important is saved to `My Drive/RVC/`, so a disconnect doesn't lose your work.

In [ ]:
#@title 1. Setup (takes about 5–8 minutes)
#@markdown Connects Google Drive, installs RVC and downloads its base models.
#@markdown A "restart session" warning about packages may appear at the end: ignore it.

RVC_REF = "81eed5e8f68b6bed1789f682fe78cdd324495afc"  # same version as the GitHub Action

import os, shutil, subprocess, sys, zipfile
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU found. Choose Runtime → Change runtime type → T4 GPU, then run this cell again.")
print(gpu.stdout.strip())
if sys.version_info[:2] != (3, 12):
    print("Warning: this RVC version targets Python 3.12, Colab has %d.%d. Setup may fail." % sys.version_info[:2])

from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/RVC")
for sub in ["dataset", "weights", "backup/logs", "export"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

RVC = Path("/content/rvc")
if not RVC.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI", str(RVC)], check=True)
subprocess.run(["git", "-C", str(RVC), "checkout", "-q", RVC_REF], check=True)

# The requirement files point at mainland-China mirrors; use the official indexes instead.
req = (RVC / "requirments_cu118_py312.txt").read_text()
req = req.replace("https://mirrors.pku.edu.cn/pypi/simple", "https://pypi.org/simple")
Path("/content/requirements-rvc.txt").write_text(req)

print("Installing packages...")
!pip install -q torch==2.7.1+cu118 torchaudio==2.7.1+cu118 torchvision==0.22.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q -r /content/requirements-rvc.txt
!apt-get -qq install -y ffmpeg > /dev/null

print("Downloading base models...")
from huggingface_hub import hf_hub_download, snapshot_download
HF_REPO = "lj1995/VoiceConversionWebUI"
snapshot_download(HF_REPO, local_dir=RVC / "assets",
                  allow_patterns=["hubert_base/*", "pretrained_v2/f0G*.pth", "pretrained_v2/f0D*.pth"])
hf_hub_download(HF_REPO, "rmvpe.pt", local_dir=RVC / "assets" / "rmvpe")
if not (RVC / "logs" / "mute").exists():
    mute_zip = hf_hub_download(HF_REPO, "mute.zip", local_dir="/content/downloads")
    with zipfile.ZipFile(mute_zip) as archive:
        archive.extractall(RVC / "logs")

# Finished models go straight to Drive.
weights = RVC / "assets" / "weights"
if weights.is_dir() and not weights.is_symlink():
    shutil.rmtree(weights)
if not weights.exists():
    weights.symlink_to(DRIVE / "weights")

# Bring back training progress saved by an earlier session.
restored = [p.name for p in (DRIVE / "backup" / "logs").iterdir() if p.is_dir()]
if restored:
    shutil.copytree(DRIVE / "backup" / "logs", RVC / "logs", dirs_exist_ok=True)
    print("Restored earlier training runs:", ", ".join(restored))

print("\nSetup complete. Run step 2.")

In [ ]:
#@title 2. Start the WebUI
#@markdown Opens RVC in a new browser tab. Training progress is backed up to Drive every 10 minutes.
#@markdown Turn on the share link only if the normal link doesn't load.
USE_SHARE_LINK = False  #@param {type:"boolean"}

import re, socket, threading, time

LOG = Path("/content/webui.log")
subprocess.run(["pkill", "-f", "webui.py"])
time.sleep(2)
args = [sys.executable, "webui.py", "--noautoopen"] + (["--colab"] if USE_SHARE_LINK else [])
webui = subprocess.Popen(args, cwd=RVC, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def backup_logs():
    """Copy new or changed training files to Drive so a disconnect can resume."""
    src, dst = RVC / "logs", DRIVE / "backup" / "logs"
    for path in src.rglob("*"):
        if not path.is_file() or path.relative_to(src).parts[0] == "mute":
            continue
        target = dst / path.relative_to(src)
        stat = path.stat()
        if target.exists() and target.stat().st_size == stat.st_size and target.stat().st_mtime >= stat.st_mtime:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)

def backup_loop():
    while webui.poll() is None:
        time.sleep(600)
        try:
            backup_logs()
        except Exception as error:
            print("Backup failed:", error)

threading.Thread(target=backup_loop, daemon=True).start()

print("Starting RVC (up to about a minute)...")
url = None
for _ in range(180):
    time.sleep(1)
    if webui.poll() is not None:
        break
    text = LOG.read_text(errors="ignore")
    if USE_SHARE_LINK:
        match = re.search(r"https://\S+\.gradio\.live", text)
        if match:
            url = match.group(0)
            break
    else:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", 7865)) == 0:
                from google.colab.output import eval_js
                url = eval_js("google.colab.kernel.proxyPort(7865)")
                break

if url:
    print("\nOpen RVC here:", url)
    print("Keep this Colab tab open while training. Logs: /content/webui.log")
else:
    print(LOG.read_text(errors="ignore")[-3000:])
    raise SystemExit("RVC did not start; see the log above. Try again with USE_SHARE_LINK turned on.")

### How to train in the WebUI

Open the **Train** tab and fill in:

| Setting | Value |
| --- | --- |
| Enter the experiment name | your model name, e.g. `MyVoice` (this becomes the model's file name) |
| Target sample rate | `40k` (use `48k` only for very high-quality recordings) |
| Model architecture version | `v2` |
| Enter the path of the training folder | `/content/drive/MyDrive/RVC/dataset/MyVoice` |
| Select the pitch extraction algorithm | `rmvpe_gpu` |
| Total training epochs | `200` to start (about 10 minutes of audio: 200–300; 30 minutes: 100–200) |
| Save frequency | `25` |
| Batch size per GPU | `8` on a T4 (lower it if you get out-of-memory errors) |
| Save only the latest '.ckpt' file | `Yes` (saves Drive space) |
| Cache all training sets to GPU memory | `Yes` if your dataset is under 10 minutes |

Then click **One-click training**. It preprocesses the audio, extracts features, trains, and builds the index.
Progress appears in `/content/webui.log`; with ~10 minutes of audio, 200 epochs takes roughly 1–2 hours on a T4.

**If Colab disconnects:** run steps 1 and 2 again, enter the *same* experiment name and settings, and click
**One-click training**. It continues from the last saved checkpoint.

**Test it:** in the **Model Inference** tab, refresh, pick your model, upload a clip and convert.

In [ ]:
#@title 3. Export the finished model
#@markdown Collects the model (.pth) and its index into `My Drive/RVC/export/<name>/` and a zip.
MODEL_NAME = "MyVoice"  #@param {type:"string"}
DOWNLOAD_ZIP = True  #@param {type:"boolean"}

if "backup_logs" in globals():
    backup_logs()
model = DRIVE / "weights" / (MODEL_NAME + ".pth")
if not model.exists():
    found = ", ".join(p.stem for p in (DRIVE / "weights").glob("*.pth")) or "none yet"
    raise SystemExit("No model named %s. Finished models: %s" % (MODEL_NAME, found))
indexes = sorted((RVC / "logs" / MODEL_NAME).glob("added_*.index"))

out = DRIVE / "export" / MODEL_NAME
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(model, out / model.name)
for index in indexes:
    shutil.copy2(index, out / index.name)
archive = shutil.make_archive(str(out), "zip", out)
print("Saved to", out)
for p in sorted(out.iterdir()):
    print("  %s (%.0f MB)" % (p.name, p.stat().st_size / 1e6))
if not indexes:
    print("No index file found: it's optional, but run 'Train feature index' in the Train tab for better quality.")
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)

### Use it with the GitHub Action

Put the `.pth` and `.index` files in the repo's `models/` folder and push them with `git` or GitHub Desktop
(the GitHub website only accepts files up to 25 MB). Then add audio to `input/` and the Action converts it.
See the repo README for details.